# 📺 YouTube Quality Analyzer — Phase 2 : Collecte des Données
## Contexte : Vidéos Éducatives Cameroun (3ème, Première, Terminale)

**Pipeline :**
1. Installation des dépendances
2. Collecte des vidéos (métadonnées) via yt-dlp
3. Collecte des commentaires par vidéo
4. Validation et rapport EDA
5. Export du dataset final

**Systèmes scolaires :** Francophone (BEPC/Bac) + Anglophone (GCE O/A-Level)

> Seed : 42 — Reproductible par tout tiers (NFR-02)

## 0. Configuration & Installation

In [ ]:
# ─── Installation des dépendances ─────────────────────────────────────────
!pip install yt-dlp langdetect --quiet

import subprocess, json, csv, time, random, logging, re
from pathlib import Path
from collections import Counter
from datetime import datetime

# Seed de reproductibilité (NFR-02)
SEED = 42
random.seed(SEED)

# Répertoires
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/validated').mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(exist_ok=True)

print('✅ Configuration OK')
print(f'   Seed : {SEED}')
print(f'   yt-dlp version :', subprocess.run(['yt-dlp', '--version'], capture_output=True, text=True).stdout.strip())

## 1. Paramètres de Collecte

In [ ]:
# ─── Paramètres modifiables ────────────────────────────────────────────────

# Systèmes cibles
SYSTEMS = ['francophone', 'anglophone']   # ou ['francophone'] seul

# Matières prioritaires (None = toutes)
TARGET_SUBJECTS = None
# Exemple pour cibler uniquement maths et SVT :
# TARGET_SUBJECTS = ['mathematiques', 'svt', 'mathematics', 'sciences']

# Niveaux cibles (None = tous)
TARGET_LEVELS = None
# Exemple pour cibler uniquement les classes d'examen :
# TARGET_LEVELS = ['troisieme', 'terminale', 'form4_5', 'upper6th']

# Nombre max de résultats par requête de recherche
MAX_PER_QUERY = 15

# Nombre max de commentaires par vidéo (limite PRD)
MAX_COMMENTS_PER_VIDEO = 500

# Filtres de qualité
QUALITY_FILTERS = {
    'min_duration_seconds': 120,
    'max_duration_seconds': 7200,
    'min_view_count': 100,
    'min_comment_count': 5,
    'min_comments_per_video': 5,
}

print('📋 Paramètres de collecte :')
print(f'   Systèmes  : {SYSTEMS}')
print(f'   Matières  : {TARGET_SUBJECTS or "toutes"}')
print(f'   Niveaux   : {TARGET_LEVELS or "tous"}')
print(f'   Max/requête : {MAX_PER_QUERY}')
print(f'   Max commentaires/vidéo : {MAX_COMMENTS_PER_VIDEO}')

## 2. Banque de Mots-Clés

In [ ]:
# ─── Mots-clés Francophone ─────────────────────────────────────────────────
KEYWORDS_FR = {
    'mathematiques': {
        'troisieme': [
            'mathématiques 3ème Cameroun',
            'cours maths troisième BEPC Cameroun',
            'exercices maths 3ème Cameroun corrigés',
            'algèbre 3ème Cameroun',
            'géométrie 3ème BEPC',
            'BEPC mathématiques Cameroun révision',
            'statistiques 3ème Cameroun',
            'maths collège Cameroun troisième',
        ],
        'premiere': [
            'mathématiques Première Cameroun',
            'cours maths Première C D Cameroun',
            'suites numériques Première Cameroun',
            'Probatoire maths Cameroun',
            'dérivées fonctions Première Cameroun',
            'probabilités Première Cameroun',
            'géométrie analytique Première Cameroun',
        ],
        'terminale': [
            'mathématiques Terminale Cameroun',
            'intégrales Terminale Cameroun',
            'Baccalauréat maths Cameroun',
            'nombres complexes Terminale Cameroun',
            'révision bac maths Cameroun',
            'équations différentielles Terminale Cameroun',
            'matrices Terminale Cameroun',
        ],
    },
    'svt': {
        'troisieme': [
            'SVT 3ème Cameroun',
            'sciences vie terre troisième Cameroun',
            'BEPC SVT Cameroun révision',
            'biologie 3ème Cameroun cours',
            'génétique 3ème Cameroun',
            'photosynthèse 3ème Cameroun',
        ],
        'premiere': [
            'SVT Première Cameroun',
            'biologie cellulaire Première Cameroun',
            'Probatoire SVT Cameroun',
            'génétique Première Cameroun',
            'immunologie Première Cameroun',
        ],
        'terminale': [
            'SVT Terminale Cameroun',
            'Baccalauréat SVT Cameroun',
            'évolution Terminale Cameroun',
            'ADN réplication Terminale Cameroun',
            'révision bac SVT Cameroun',
        ],
    },
    'physique_chimie': {
        'troisieme': [
            'physique chimie 3ème Cameroun',
            'BEPC physique Cameroun',
            'électricité 3ème Cameroun',
            'optique 3ème Cameroun',
        ],
        'premiere': [
            'physique Première Cameroun',
            'Probatoire physique Cameroun',
            'chimie organique Première Cameroun',
            'électrocinétique Première Cameroun',
        ],
        'terminale': [
            'physique Terminale Cameroun',
            'Baccalauréat physique Cameroun',
            'électromagnétisme Terminale Cameroun',
            'révision bac physique Cameroun',
        ],
    },
    'francais_litterature': {
        'troisieme': [
            'français 3ème Cameroun',
            'BEPC français Cameroun',
            'dissertation 3ème Cameroun',
            'commentaire texte 3ème Cameroun',
        ],
        'premiere': [
            'français Première Cameroun',
            'Probatoire français Cameroun',
            'commentaire composé Première Cameroun',
            'littérature Première Cameroun',
        ],
        'terminale': [
            'français Terminale Cameroun',
            'Baccalauréat français Cameroun',
            'oeuvres bac français Cameroun',
            'révision bac français Cameroun',
        ],
    },
    'histoire_geo': {
        'troisieme': [
            'histoire géographie 3ème Cameroun',
            'BEPC histoire géo Cameroun',
            'géographie Cameroun troisième',
        ],
        'premiere': [
            'histoire Première Cameroun',
            'Probatoire histoire géo Cameroun',
            'géopolitique Première Cameroun',
        ],
        'terminale': [
            'histoire Terminale Cameroun',
            'Baccalauréat histoire géo Cameroun',
            'révision bac histoire Cameroun',
        ],
    },
    'economie': {
        'premiere': [
            'économie Première Cameroun',
            'Probatoire économie Cameroun',
            'microéconomie Première Cameroun',
        ],
        'terminale': [
            'économie Terminale Cameroun',
            'Baccalauréat économie Cameroun',
            'macroéconomie Terminale Cameroun',
        ],
    },
}

# ─── Mots-clés Anglophone ──────────────────────────────────────────────────
KEYWORDS_EN = {
    'mathematics': {
        'form4_5': [
            'mathematics Form 4 Cameroon',
            'GCE O-Level maths Cameroon',
            'algebra Form 4 Cameroon',
            'geometry Form 5 Cameroon',
            'O-Level mathematics revision Cameroon',
        ],
        'lower6th': [
            'A-Level mathematics Lower 6 Cameroon',
            'calculus Lower 6th Cameroon',
            'pure mathematics A-Level Cameroon',
            'GCE A-Level maths Cameroon',
        ],
        'upper6th': [
            'A-Level mathematics Upper 6 Cameroon',
            'integration Upper 6th Cameroon',
            'GCE A-Level revision maths Cameroon',
            'A-Level maths past questions Cameroon',
        ],
    },
    'sciences': {
        'form4_5': [
            'biology Form 4 Cameroon',
            'chemistry O-Level Cameroon',
            'physics Form 5 Cameroon',
            'GCE O-Level science Cameroon',
        ],
        'lower6th': [
            'biology Lower 6th Cameroon',
            'chemistry A-Level Cameroon',
            'GCE A-Level biology Cameroon',
            'genetics A-Level Cameroon',
        ],
        'upper6th': [
            'biology Upper 6th Cameroon',
            'chemistry A-Level revision Cameroon',
            'GCE A-Level science revision Cameroon',
            'A-Level past papers science Cameroon',
        ],
    },
    'english_literature': {
        'form4_5': [
            'English language Form 4 Cameroon',
            'GCE O-Level English Cameroon',
            'essay writing O-Level Cameroon',
        ],
        'lower6th': [
            'English literature Lower 6th Cameroon',
            'A-Level English Cameroon',
            'GCE A-Level English Cameroon',
        ],
        'upper6th': [
            'English literature Upper 6th Cameroon',
            'GCE A-Level literature Cameroon',
            'A-Level past papers English Cameroon',
        ],
    },
    'history_geography': {
        'form4_5': [
            'history Form 4 Cameroon',
            'geography O-Level Cameroon',
            'GCE O-Level history Cameroon',
        ],
        'upper6th': [
            'history A-Level Cameroon',
            'GCE A-Level history Cameroon',
            'A-Level history past papers Cameroon',
        ],
    },
    'economics': {
        'lower6th': [
            'economics Lower 6th Cameroon',
            'A-Level economics Cameroon',
        ],
        'upper6th': [
            'economics Upper 6th Cameroon',
            'GCE A-Level economics Cameroon',
            'A-Level economics past papers Cameroon',
        ],
    },
}

ALL_KEYWORDS = {'francophone': KEYWORDS_FR, 'anglophone': KEYWORDS_EN}

# Compter les requêtes totales
total_queries = sum(
    len(queries)
    for sys_kw in ALL_KEYWORDS.values()
    for subj_kw in sys_kw.values()
    for queries in subj_kw.values()
)
print(f'✅ Banque de mots-clés chargée : {total_queries} requêtes de recherche')

## 3. Collecte des Vidéos (Métadonnées)

In [ ]:
def parse_duration(d):
    if d is None: return 0
    if isinstance(d, (int, float)): return int(d)
    try:
        parts = str(d).split(':')
        if len(parts) == 3: return int(parts[0])*3600 + int(parts[1])*60 + int(parts[2])
        if len(parts) == 2: return int(parts[0])*60 + int(parts[1])
        return int(d)
    except: return 0

def passes_filter(v):
    dur   = parse_duration(v.get('duration'))
    views = v.get('view_count') or 0
    # comment_count absent des résultats de recherche YouTube — filtré à l'étape 4
    if dur   < QUALITY_FILTERS['min_duration_seconds']:  return False, f'durée {dur}s'
    if dur   > QUALITY_FILTERS['max_duration_seconds']:  return False, f'durée {dur}s'
    if views < QUALITY_FILTERS['min_view_count']:        return False, f'vues {views}'
    return True, ''

def search_youtube(query, max_results, system, level, subject):
    cmd = ['yt-dlp', f'ytsearch{max_results}:{query}',
           '--skip-download', '--print-json', '--no-warnings',
           '--ignore-errors']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60, encoding='utf-8')
    except subprocess.TimeoutExpired:
        print(f'  ⏱️  Timeout: {query}')
        return []
    rows = []
    for line in r.stdout.strip().split('\n'):
        if not line.strip(): continue
        try:
            d = json.loads(line)
            ok, reason = passes_filter(d)
            if ok and d.get('id'):
                rows.append({
                    'video_id': d.get('id',''),
                    'title': d.get('title',''),
                    'channel_name': d.get('uploader') or d.get('channel',''),
                    'channel_id': d.get('channel_id',''),
                    'published_at': d.get('upload_date',''),
                    'duration_seconds': parse_duration(d.get('duration')),
                    'view_count': d.get('view_count') or 0,
                    'like_count': d.get('like_count') or 0,
                    'comment_count': d.get('comment_count') or 0,
                    'language': 'fr' if system == 'francophone' else 'en',
                    'system': system,
                    'level': level,
                    'subject': subject,
                    'search_query': query,
                    'url': f"https://www.youtube.com/watch?v={d.get('id','')}",
                })
            elif not ok:
                print(f'   ⛔ Filtré ({reason}) : {d.get("title","")[:50]}')
        except: continue
    return rows

# ─── Collecte principale ───────────────────────────────────────────────────
all_videos = []
seen_ids   = set()
n_queries  = 0

for system in SYSTEMS:
    kw_bank = ALL_KEYWORDS[system]
    for subject, levels_kw in kw_bank.items():
        if TARGET_SUBJECTS and subject not in TARGET_SUBJECTS:
            continue
        for level, queries in levels_kw.items():
            if TARGET_LEVELS and level not in TARGET_LEVELS:
                continue
            for query in queries:
                n_queries += 1
                print(f'[{n_queries}] 🔍 {query}')
                videos = search_youtube(query, MAX_PER_QUERY, system, level, subject)
                for v in videos:
                    if v['video_id'] not in seen_ids:
                        seen_ids.add(v['video_id'])
                        all_videos.append(v)
                        print(f'   ✅ {v["title"][:55]} | vues={v["view_count"]}')
                delay = random.uniform(3, 7)
                time.sleep(delay)

print(f'\n📊 Vidéos uniques collectées : {len(all_videos)}')

# Sauvegarde
VIDEO_COLS = ['video_id','title','channel_name','channel_id','published_at',
              'duration_seconds','view_count','like_count','comment_count',
              'language','system','level','subject','search_query','url']
with open('data/raw/videos.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=VIDEO_COLS, extrasaction='ignore')
    w.writeheader(); w.writerows(all_videos)
print('💾 Sauvegardé : data/raw/videos.csv')

## 4. Collecte des Commentaires

In [ ]:
COMMENT_COLS = ['video_id','comment_id','text','author','author_likes',
                'reply_count','published_at','is_reply','parent_id','language_detected']

checkpoint_path = Path('data/raw/.checkpoint_comments.json')
processed_ids = set()
if checkpoint_path.exists():
    with open(checkpoint_path) as f:
        processed_ids = set(json.load(f).get('processed_ids', []))
    print(f'🔄 Reprise checkpoint : {len(processed_ids)} vidéos déjà traitées')

output_exists = Path('data/raw/comments.csv').exists() and bool(processed_ids)
mode = 'a' if output_exists else 'w'

total_comments = 0

with open('data/raw/comments.csv', mode, newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COMMENT_COLS, extrasaction='ignore')
    if mode == 'w': writer.writeheader()

    for i, video in enumerate(all_videos, 1):
        vid_id = video['video_id']
        if vid_id in processed_ids:
            continue

        print(f'[{i}/{len(all_videos)}] 💬 {video["title"][:55]}')

        cmd = ['yt-dlp', f'https://www.youtube.com/watch?v={vid_id}',
               '--skip-download', '--write-comments', '--print-json',
               '--no-warnings', '--ignore-errors',
               '--extractor-args', f'youtube:max_comments={MAX_COMMENTS_PER_VIDEO},max_comment_depth=2']
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=120, encoding='utf-8')
            lines = [l for l in r.stdout.strip().split('\n') if l.strip()]
            if not lines: raise ValueError('no output')
            data = json.loads(lines[-1])
            comments_raw = data.get('comments') or []
            rows = []
            for c in comments_raw[:MAX_COMMENTS_PER_VIDEO]:
                text = (c.get('text') or '').strip()
                if not text: continue
                parent = c.get('parent', 'root')
                is_reply = parent != 'root'
                ts = c.get('timestamp')
                pub = datetime.utcfromtimestamp(ts).isoformat() if isinstance(ts, (int,float)) else ''
                rows.append({
                    'video_id': vid_id,
                    'comment_id': c.get('id',''),
                    'text': text,
                    'author': c.get('author',''),
                    'author_likes': c.get('like_count') or 0,
                    'reply_count': c.get('reply_count') or 0,
                    'published_at': pub,
                    'is_reply': is_reply,
                    'parent_id': parent if is_reply else '',
                    'language_detected': '',
                })
            if len(rows) >= QUALITY_FILTERS['min_comments_per_video']:
                writer.writerows(rows); f.flush()
                total_comments += len(rows)
                print(f'   ✅ {len(rows)} commentaires')
            else:
                print(f'   ⛔ Trop peu de commentaires : {len(rows)}')
        except Exception as e:
            print(f'   ⚠️  Erreur : {e}')

        processed_ids.add(vid_id)
        with open(checkpoint_path, 'w') as cp:
            json.dump({'processed_ids': list(processed_ids),
                       'updated': datetime.utcnow().isoformat()}, cp)

        time.sleep(random.uniform(5, 12))

print(f'\n✅ Collecte terminée — {total_comments} commentaires dans data/raw/comments.csv')

## 5. Validation & EDA

In [ ]:
# ─── Chargement ────────────────────────────────────────────────────────────
with open('data/raw/videos.csv', encoding='utf-8') as f:
    videos_df = list(csv.DictReader(f))
with open('data/raw/comments.csv', encoding='utf-8') as f:
    comments_df = list(csv.DictReader(f))

print(f'Vidéos : {len(videos_df)}')
print(f'Commentaires : {len(comments_df)}')

# ─── Distribution ──────────────────────────────────────────────────────────
print('\n─── Distribution par système ───')
for k, v in Counter(r['system'] for r in videos_df).items():
    print(f'  {k:15} : {v}')

print('\n─── Distribution par niveau ───')
for k, v in Counter(r['level'] for r in videos_df).most_common():
    print(f'  {k:15} : {v}')

print('\n─── Distribution par matière ───')
for k, v in Counter(r['subject'] for r in videos_df).most_common():
    print(f'  {k:20} : {v}')

# ─── Commentaires/vidéo ────────────────────────────────────────────────────
cpv = Counter(c['video_id'] for c in comments_df)
vals = list(cpv.values())
print(f'\n─── Commentaires par vidéo ───')
print(f'  Moyenne : {sum(vals)/len(vals):.1f}')
print(f'  Min     : {min(vals)}')
print(f'  Max     : {max(vals)}')

# ─── Validation colonnes A1 ────────────────────────────────────────────────
required = {'video_id', 'text', 'author_likes', 'reply_count'}
present  = set(comments_df[0].keys()) if comments_df else set()
missing  = required - present
print(f'\n─── Compatibilité Agent A1 ───')
if missing:
    print(f'  ❌ Colonnes manquantes : {missing}')
else:
    print(f'  ✅ Toutes les colonnes A1 présentes')
    print(f'  ✅ Dataset prêt pour le pipeline LangGraph')

## 6. Export Dataset Final

In [ ]:
# ─── Exclure vidéos avec trop peu de commentaires ──────────────────────────
valid_ids = {vid for vid, cnt in cpv.items()
             if cnt >= QUALITY_FILTERS['min_comments_per_video']}

final_videos   = [v for v in videos_df   if v['video_id'] in valid_ids]
final_comments = [c for c in comments_df if c['video_id'] in valid_ids]

# Sauvegarde
with open('data/validated/videos_final.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=VIDEO_COLS, extrasaction='ignore')
    w.writeheader(); w.writerows(final_videos)

with open('data/validated/comments_final.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=COMMENT_COLS, extrasaction='ignore')
    w.writeheader(); w.writerows(final_comments)

print('✅ Dataset final exporté :')
print(f'   data/validated/videos_final.csv   — {len(final_videos)} vidéos')
print(f'   data/validated/comments_final.csv — {len(final_comments)} commentaires')
print()
print('🔜 Prochaine étape : alimenter Agent A1 (Loader/Validator) du pipeline LangGraph')
print(f'   Input A1 : data/validated/comments_final.csv')